<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/vllm/vllm_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vLLM on Colab L4 / A100 / G4

Installs vLLM (the packages are cached on Google Drive), serves a model that fits the GPU with tool calling and an OpenAI-compatible API, reachable from your own machine over Tailscale.

Runtime -> Change runtime type -> **L4**, **A100** or **G4 GPU**. vLLM serves safetensors checkpoints (bf16, FP8, NVFP4, AWQ...), not GGUF files; for GGUF use `llama_cpp_gpu.ipynb`. A T4 is not supported by recent vLLM builds; use the llama.cpp notebook there.

## 1. Detect GPU, CPU, and RAM

In [ ]:
import os, re, subprocess
os.environ["PATH"] += ":/usr/local/cuda/bin"

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
if smi.returncode or not smi.stdout.strip():
    raise RuntimeError("No NVIDIA GPU found. Runtime -> Change runtime type -> choose an L4, A100 or G4 GPU.")
GPU_NAME, mem_mib, cc = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(",")]
VRAM_GB   = int(float(mem_mib)) / 1024
CUDA_ARCH = cc.replace(".", "")                      # 8.0 -> 80, 8.9 -> 89, 12.0 -> 120
TIER = "xl" if VRAM_GB >= 80 else "l" if VRAM_GB >= 36 else "m" if VRAM_GB >= 20 else "s"
print(f"GPU: {GPU_NAME}: {VRAM_GB:.0f} GB VRAM, compute capability {cc} (sm_{CUDA_ARCH}), size tier '{TIER}'")

def _host():
    model, cores, threads, sockets = "unknown CPU", 0, 0, set()
    cur_phys = cur_core = None
    core_ids = set()
    try:
        text = open("/proc/cpuinfo", encoding="utf-8", errors="replace").read()
    except OSError:
        text = ""
    for line in text.splitlines():
        if not line.strip():
            if cur_phys is not None and cur_core is not None:
                core_ids.add((cur_phys, cur_core))
            cur_phys = cur_core = None
            continue
        if ":" not in line:
            continue
        k, v = [x.strip() for x in line.split(":", 1)]
        if k == "model name" and model == "unknown CPU":
            model = v
        elif k == "processor":
            threads += 1
        elif k == "physical id":
            cur_phys = v; sockets.add(v)
        elif k == "core id":
            cur_core = v
    if cur_phys is not None and cur_core is not None:
        core_ids.add((cur_phys, cur_core))
    threads = threads or (os.cpu_count() or 1)
    cores = len(core_ids) or threads
    mem = {}
    try:
        for line in open("/proc/meminfo", encoding="utf-8", errors="replace"):
            if ":" not in line: continue
            k, rest = line.split(":", 1)
            parts = rest.split()
            if parts:
                try: mem[k] = int(parts[0])
                except ValueError: pass
    except OSError:
        pass
    total_kb = mem.get("MemTotal", 0)
    avail_kb = mem.get("MemAvailable", mem.get("MemFree", 0))
    to_gb = lambda kb: kb / 1_000_000
    CPU_MODEL, CPU_CORES, CPU_THREADS = model, cores, threads
    RAM_GB, RAM_AVAIL_GB = to_gb(total_kb), to_gb(avail_kb)
    sock = f" ({len(sockets)} sockets)" if len(sockets) > 1 else ""
    print(f"CPU: {CPU_MODEL} — {CPU_CORES} cores / {CPU_THREADS} threads{sock}")
    print(f"RAM: {RAM_GB:.1f} GB total, {RAM_AVAIL_GB:.1f} GB available ({(RAM_GB-RAM_AVAIL_GB):.1f} GB used)")
    try:
        u = os.statvfs("/content")
        print(f"Disk (/content): {u.f_bavail*u.f_frsize/1e9:.1f} GB free / {u.f_blocks*u.f_frsize/1e9:.1f} GB total")
    except OSError:
        pass
    return CPU_MODEL, CPU_CORES, CPU_THREADS, RAM_GB, RAM_AVAIL_GB

CPU_MODEL, CPU_CORES, CPU_THREADS, RAM_GB, RAM_AVAIL_GB = _host()


## 2. Config (the model is picked from the GPU's memory; override any value)

In [ ]:
# Qwen3.8-27B needs vLLM 0.17 or newer (pip installs the latest). Checkpoint sizes: bf16 ~55.6 GB, FP8 ~30.9 GB, NVFP4 (Blackwell) ~21.6 GB.
PRESETS = {
    # 80+ GB: G4 (96 GB), A100 80 GB. FP8 leaves lots of room for context; use "Qwen/Qwen3.8-27B" for full bf16.
    "xl": dict(model="Qwen/Qwen3.8-27B-FP8", len=65536, util=0.90, tools="qwen3_coder", reasoning="qwen3"),
    # 36-79 GB: A100 40 GB. 30.9 GB of weights, so the context stays short.
    "l":  dict(model="Qwen/Qwen3.8-27B-FP8", len=16384, util=0.92, tools="qwen3_coder", reasoning="qwen3"),
    # 20-35 GB: L4 (24 GB). The 27B FP8 does not fit; 14B FP8 (16.3 GB) does.
    "m":  dict(model="Qwen/Qwen3-14B-FP8",   len=16384, util=0.90, tools="hermes",      reasoning="qwen3"),
    # under 20 GB: a small model, if your vLLM build supports the GPU at all
    "s":  dict(model="Qwen/Qwen3-4B",        len=8192,  util=0.90, tools="hermes",      reasoning="qwen3"),
}

# Override to choose your own (any Hugging Face safetensors model vLLM supports).
MODEL_ID         = None
MAX_MODEL_LEN    = None
TOOL_PARSER      = None   # vLLM tool-call parser; Qwen3.5/3.8 use qwen3_coder (some builds also offer qwen3_xml), Qwen3 uses hermes
REASONING_PARSER = None   # "qwen3" separates the thinking text from the answer
REFRESH_WHEELS   = False  # True = re-download the packages and re-save them on Drive

p = PRESETS[TIER]
MODEL_ID         = MODEL_ID or p["model"]
MAX_MODEL_LEN    = MAX_MODEL_LEN or p["len"]
TOOL_PARSER      = TOOL_PARSER or p["tools"]
REASONING_PARSER = REASONING_PARSER or p["reasoning"]
GPU_UTIL         = p["util"]
print("Model:", MODEL_ID, "| context:", MAX_MODEL_LEN, "| tool parser:", TOOL_PARSER)

## 3. Keep the Hugging Face cache on Google Drive (the model is downloaded once and reused)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
HF_CACHE = "/content/drive/MyDrive/llm_models/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE          # inherited by the vllm server started below
print("Hugging Face cache:", HF_CACHE)

## 4. Install vLLM (the packages are saved on Google Drive after the first install)

In [ ]:
import sys, os, glob, subprocess

WHEELS = f"/content/drive/MyDrive/llm_models/builds/vllm_gpu_py{sys.version_info.major}{sys.version_info.minor}"
PKGS = ["vllm"]

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    open("/content/pip_install.log", "a").write(r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError("pip failed:\n" + (r.stdout + r.stderr)[-3000:])

open("/content/pip_install.log", "w").close()
cached = bool(glob.glob(f"{WHEELS}/*.whl")) and not REFRESH_WHEELS
if cached:
    print("Installing from the wheels saved on Drive...")
    try:
        pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
    except RuntimeError as e:
        print("Cached install failed, rebuilding the wheel cache:\n", str(e)[:500])
        cached = False
if not cached:
    print("Downloading packages and saving them to Drive (slow, one time; several GB including PyTorch)...")
    os.makedirs(WHEELS, exist_ok=True)
    pip("wheel", "-q", "-w", WHEELS, *PKGS)
    pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
print("Done. Wheels:", len(glob.glob(f"{WHEELS}/*.whl")))
print(subprocess.run([sys.executable, "-m", "pip", "show", "vllm"], capture_output=True, text=True).stdout.split("\n")[1])

## 5. Start the server (the first start downloads the model, then loads it and compiles kernels, which can take several minutes)

In [ ]:
import subprocess, time, requests, secrets

import os
KEY_FILE = "/content/drive/MyDrive/llm_models/hub/api_key"   # stable key on private Drive, shared with the hub notebooks
if os.path.exists(KEY_FILE) and len(open(KEY_FILE).read().strip()) >= 24:
    API_KEY = open(KEY_FILE).read().strip()
else:
    os.makedirs(os.path.dirname(KEY_FILE), exist_ok=True)
    API_KEY = secrets.token_urlsafe(32)
    open(KEY_FILE, "w").write(API_KEY + "\n")

cmd = ["vllm", "serve", MODEL_ID,
       "--max-model-len", str(MAX_MODEL_LEN),
       "--gpu-memory-utilization", str(GPU_UTIL),
       "--max-num-seqs", "16",
       "--enable-auto-tool-choice", "--tool-call-parser", TOOL_PARSER,
       "--host", "127.0.0.1", "--port", "8000", "--api-key", API_KEY]
if REASONING_PARSER:
    cmd += ["--reasoning-parser", REASONING_PARSER]

server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(900):                      # up to ~30 min
    if server.poll() is not None:
        print("server exited:\n", open("/content/vllm.log").read()[-5000:]); break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/vllm.log").read()[-4000:])

## 6. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    t0 = time.time()
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions",
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"model": MODEL_ID, "messages": [{"role": "user", "content": prompt}],
                            "max_tokens": max_tokens, "temperature": temperature}, timeout=900)
    r.raise_for_status()
    j = r.json(); m = j["choices"][0]["message"]
    n = j.get("usage", {}).get("completion_tokens")
    if n: print(f"[{n} tokens, {n / (time.time() - t0):.1f} tokens/s including prompt processing]")
    thinking = m.get("reasoning_content") or m.get("reasoning")
    return (("[thinking]\n" + thinking + "\n[answer]\n") if thinking else "") + (m.get("content") or "")

print(chat("Explain what vLLM does in two sentences."))

## 7. Connect your machine over Tailscale

This joins the Colab VM to your Tailscale tailnet and publishes the server over HTTPS at `https://colab-vllm.<your-tailnet>.ts.net`, reachable only from devices in your tailnet. Run your harness (dsh or any OpenAI-compatible client) **on your own machine** in the same tailnet, pointed at the Base URL and API key printed below. Nothing harness-related runs in this notebook.

Create a **reusable + ephemeral** auth key at https://login.tailscale.com/admin/settings/keys. The first run asks for it and saves it to private Drive (`MyDrive/llm_models/hub/tailscale_authkey`); later sessions reuse it. The API key is also kept on Drive, and the hostname stays the same, so your harness settings keep working across sessions.

In [ ]:
# Uses the hub package's Tailscale helper (standard library only; nothing to install).
import os, sys, subprocess
HUB_DIR = "/content/hub_repo"
if not os.path.isdir(f"{HUB_DIR}/colab_llm_hub"):
    subprocess.run(f"rm -rf {HUB_DIR} && git clone -q --depth 1 https://github.com/Biegle-W/google_colab_llm {HUB_DIR}",
                   shell=True, check=True)
if HUB_DIR not in sys.path:
    sys.path.insert(0, HUB_DIR)
from colab_llm_hub import tunnel, provider

DRIVE_DIR   = "/content/drive/MyDrive/llm_models"
PORT        = 8000
TS_HOSTNAME = "colab-vllm"   # this server's name on your tailnet
TS_FUNNEL   = False   # True = also reachable from the public internet (the API key still applies)

ts_key = tunnel.load_or_ask_tailscale_authkey(f"{DRIVE_DIR}/hub/tailscale_authkey")
ts_proc, BASE = tunnel.start_tailscale(PORT, ts_key, hostname=TS_HOSTNAME, drive_dir=DRIVE_DIR, funnel=TS_FUNNEL)
del ts_key   # the key stays only in the private Drive file
MODEL_NAME = MODEL_ID
print("Base URL:", BASE + "/v1")
print("API key :", API_KEY)
print("Model   :", MODEL_NAME)
print()
print("On your machine (same tailnet), any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print(f'curl {BASE}/v1/chat/completions -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" '
      f'-d \'{{"model": "{MODEL_NAME}", "messages": [{{"role": "user", "content": "Hello"}}]}}\'')

## Notes
- Qwen3.8-27B is a hybrid thinking model; the reasoning parser separates the thinking text from the answer.
- The model is picked from the GPU's memory (section 2). Large FP8 checkpoints are downloaded to Google Drive on the first run (about 31 GB for the 27B) and read from Drive afterwards, so the first load can be slow.
- The vLLM packages are cached on Drive (`MyDrive/llm_models/builds`) as wheels. Set `REFRESH_WHEELS = True` to update them. Install errors are in `/content/pip_install.log`; server errors in `/content/vllm.log`.
- A100 GPUs have no native FP8 math, so vLLM runs FP8 weights through a slower weight-only path there; L4 and G4 handle FP8 natively.
- vLLM's newest Blackwell (G4) support depends on the version pip installs; if the server fails to start on a G4, check `/content/vllm.log`.
- The API key (`MyDrive/llm_models/hub/api_key`) and Tailscale auth key are kept on private Drive and the Tailscale hostname is fixed, so the Base URL and key stay the same across sessions. If the previous session's node is still listed in your tailnet, Tailscale adds a suffix (`-1`); remove the old node to keep the name.